# Demo 2: Model Analysis

Here, we expose Whisper and analyse:
1. encoder and cross attention for unattacked and attacked audio,
2. how attention changes over the snippet's training iterations,
3. P(eot) at every decoder block (decoder bypass / logit lens),
4. causal mediation analysis (CMA) over decoder blocks,
5. the snippet vs a mean-amplitude filler of the same length, and
6. the snippet's frequency profile.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch

from asrvuln import config, data, models, audio, viz
from asrvuln.analysis import attention as attn, causal, logit_lens, spectral

device = models.get_device()

## Setup

In [ ]:
MODEL_SIZE = "tiny.en"
SNIPPET_PATH = config.SNIPPETS_DIR / "complete_snippets.pt"   # scripts/train_attack.py --mode complete

if not SNIPPET_PATH.exists():
    raise FileNotFoundError(f"{SNIPPET_PATH} not found. Train one first: python scripts/train_attack.py --mode complete")

model, processor = models.load_hf_whisper(MODEL_SIZE, device)     # downloads weights on first use
dataset = data.load_tedlium("test", n=100)

snippets = torch.load(SNIPPET_PATH, weights_only=True)            # (iterations + 1, N), best last
snippet = snippets[-1]
example = dataset[10]["audio"]
attacked = torch.cat([snippet, example])
print(dataset[10]["text"])

## 1. Attention in Unattacked and Attacked Audio

In [ ]:
BLOCK, HEAD = 0, 2

for name, wave in {"clean": example, "attacked": attacked}.items():
    decoded, _, _, e_attns, _, c_attns = attn.inference(wave, model, processor, skip_special_tokens=True)
    print(f"{name}: {decoded!r}")
    limit = attn.encoder_limit(wave.size(0))
    viz.plot_spikes(wave, attn.stack_cross_attns(c_attns, limit)[BLOCK, HEAD], lim=0.4, cmap="magma")

In [ ]:
# full encoder attention grid for the attacked example (tiny.en: 4 blocks x 6 heads)
_, _, _, e_attns, _, _ = attn.inference(attacked, model, processor, skip_special_tokens=True)
enc = attn.stack_encoder_attns(e_attns, attn.encoder_limit(attacked.size(0)))
viz.plot_attns(enc, rows=4, cols=6, figsize=(36, 22))

## 2. Attention over Training Iterations

In [ ]:
frames = attn.attention_over_snippets(model, processor, snippets, example, kind="cross", block=BLOCK, head=HEAD)
html, anim = viz.plot_attns_iters_anim(attn.smoothen(frames, 10), figsize=(10, 10),
                                       vmin=-0.05, vmax=0.05, cmap="magma", interval=50)
# anim.save(config.FIGURES_DIR / "cross_attn.mp4") to keep it
html

## 3. Decoder Bypass

In [ ]:
for name, wave in {"clean": example, "attacked": attacked}.items():
    viz.plot_eos_by_layer(logit_lens.decoder_bypass(model, processor, wave), title=name)

## 4. Causal Mediation Analysis

Corrupt the decoder token embeddings with noise, then restore one block's clean output at one step and
measure how much P(EOT) recovers.

In [ ]:
clean_model, _ = models.load_hf_whisper(MODEL_SIZE, device, output_attentions=False)
corrupted_model, _ = models.load_hf_whisper(MODEL_SIZE, device, output_attentions=False)

wave = data.load_tedlium("test", n=100, seed=1)[20]["audio"]
effects = causal.run_causal_trace(clean_model, corrupted_model, processor, wave, iters=10, alpha=3)
viz.plot_effects(effects)

## 5. Snippet vs Mean Filler Differential

In [ ]:
filler = audio.mean_amplitude_filler(snippet, example)
for name, wave in {"snippet": attacked, "mean-amplitude filler": torch.cat([filler, example])}.items():
    decoded, _, _, _, _, c_attns = attn.inference(wave, model, processor, skip_special_tokens=True)
    print(f"{name}: {decoded!r}")
    viz.plot_spikes(wave, attn.stack_cross_attns(c_attns, attn.encoder_limit(wave.size(0)))[BLOCK, HEAD], lim=0.5)

## 6. Frequency Profile

In [ ]:
speech = spectral.mean_mel_energy(data.load_tedlium("train", n=200))
viz.plot_mel_band_energy({"speech (mean over 200 clips)": speech, "snippet": spectral.snippet_mel_energy(snippet)})